# G20 — Is there forecast value in text × audio-visual non-additivity? (5-fold CV, train+val; test untouched)

**Question.** Does B's clip-IV emotion depend on how what is said (X) combines with how it is expressed (Z), beyond
what each source gives on its own? This is a gate before any method work. It measures the **forecast value of
non-additivity in the logits**. It is not a PID synergy estimate, and passing it would not show that a model has
learned pragmatics.

* X = text features of clips I–III (taken before the sum that forms the speech token).
* Z = audio, voice (who-speaks cues), faces and scene of clips I–III.

**Models (two families, the same 5 episode folds as G8b–G19).**

| Arm | Logits | Role |
|---|---|---|
| `Additive` | a_y(X) + b_y(Z), both branches trained **jointly** by CE on the sum, no exchange of information | main control |
| `Local` | Additive + Σ_t r_y(x_t, a_t): text × audio of the **same clip**, r shared over clips | **main test** |
| `Window` | any X–Z interaction in the window | secondary |
| `LateFusion` | sum of the log-probabilities of two separately trained single-source models | control, reported only |

* Neural family: the X branch is a small Transformer over the 3 text tokens. The Z branch is RoleNet without the text
  term in its speech tokens (aux heads and modality dropout kept; they see Z only). r is a pure bilinear form
  (x̃_t ⊙ ã_t → 7 logits, no bias terms, zero at init; set to 0 for clips without audio). `Window` = full RoleNet
  (G14 `Full`). The same optimiser, early stopping (dev UAR on 5 held-out training episodes) and seeds are used for
  every arm. Each seed's logits are temperature-scaled on the early-stopping episodes; the ensemble is the mean of the
  calibrated probabilities of the 10 seeds.
* Logistic family: multinomial LR on the concatenated features (X: text PCA-32 per clip; Z: audio PCA-32, scene
  PCA-32, mean HSEmotion + face flag, voice cues, audio-found flag per clip). `Local` adds Σ_t vec(x̃_t ã_tᵀ) with
  x̃, ã = PCA-8 of text and audio shared over clips. `Window` adds the outer product of PCA-8 of the whole X block and
  PCA-8 of the whole Z block. C is chosen per arm by inner 5-fold source-grouped CV on NLL; the temperature is fitted
  on the same inner out-of-fold logits. All PCA, scaling, C choice, early stopping and calibration stay inside the
  training part of each outer fold.

**Decision (fixed before running).** Δ_NLL = NLL(Additive) − NLL(Local) on the pooled out-of-fold predictions;
positive means the interaction model is better. 95% CI: neural by a two-level bootstrap over seeds and source folders;
LR by a bootstrap over source folders; paired draws.

| Main test (Local variant only) | Decision |
|---|---|
| CI of Δ_NLL above 0 in **both** families | a method pilot is allowed |
| only one family | stop developing this direction on Hi-EF |
| neither family | stop developing this direction on Hi-EF |

A failed test means *no sufficiently strong evidence with these data and model classes*; it does **not** show that
the true distribution has no interaction. The `Window` variant, UAR and `LateFusion` are reported but cannot replace
the main criterion; the two families are compared only on the same variant.

**T2 (EMAP, Hessel & Lee 2020) on the logits of full RoleNet.** The projection
f̂(x_i, z_i) = mean_j f(x_i, z_j) + mean_j f(x_j, z_i) − mean_jk f(x_j, z_k) is computed over the cross-pairs inside
each evaluation fold (and inside the early-stopping episodes, for its own temperature). The cross-pairs only serve the
projection of the model; they are not labelled counterfactual conversations. If EMAP barely lowers the score, the
reading is: *no added forecast value of RoleNet's non-additive part has been seen*. If the main test passes but EMAP
shows no drop, the pilot is still allowed, but the gains of full RoleNet are not attributed to interaction.

In [ ]:
# ======== CONFIG ========
import os


def first_existing(*paths):
    for p in paths:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"none of {paths}")


DATASET_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-dataset"
FEATURES_DIR = "/kaggle/input/datasets/ptrnghieu/hi-ef-features-v2"
SPLIT_CSV = first_existing("/kaggle/input/datasets/ptrnghieu/hi-ef-split/source_folder_split_seed42.csv",
                           "/kaggle/input/hi-ef-split/source_folder_split_seed42.csv")
G8A_DIR = first_existing("/kaggle/input/datasets/ptrnghieu/g8a-features", "/kaggle/input/g8a-features")
OUT_DIR = "/kaggle/working"

N_OUTER, N_INNER_DEV = 5, 5
SEEDS = [42, 123, 456, 7, 11, 19, 23, 31, 37, 43]    # as G14 / G19
LR, WEIGHT_DECAY = 1e-4, 1e-5                # B1 settings (only needed by the shared model cell)
FC_EPOCHS, PATIENCE, FC_BATCH = 50, 8, 32
RN = dict(D=128, heads=4, layers=2, dropout=0.2, lr=3e-4, wd=1e-2, epochs=80, patience=12, batch=64,
          aux_w=0.3, a_w=0.3, p_drop_ctx=0.3, p_drop_face=0.15)     # G8b, unchanged
PCA_DIM, MAXF, MAXF_POOL = 128, 24, 32
SAME_PERSON_COS, DOMINANT_MIN_FRAC = 0.45, 0.25
VOICE_SAME_COS = 0.35
DEBUG_PER_EPISODE = None     # e.g. 6 for a quick smoke test

FULL = dict(role=True, faces=True, ctx=True, aux=True, mdrop=True)
BASE = dict(readout='query', nq=1, q_readonly=False, role_emb=True, clip_emb=True, absent='token',
            drop=(), speech_parts=('text', 'audio', 'voice'), clips=(0, 1, 2), seed_offset=0)
ARMS = [("Full", 'tok', BASE)]          # only so that the shared cells run; the G20 arms are defined below
ZCFG = {**BASE, 'speech_parts': ('audio', 'voice')}      # RoleNet without the text term = the Z branch
N_BOOT, PCA_K, PROD_K, R_DIM = 1000, 32, 8, 32
LR_CS = [0.003, 0.01, 0.03, 0.1, 0.3, 1.0]
EXPERIMENTS = [a for a in ARMS if a[1] == 'role']

In [ ]:
import os, json, math, random, time
import numpy as np
import pandas as pd

EMO = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
POL = ['positive', 'neutral', 'negative']
E2I = {e: i for i, e in enumerate(EMO)}
P2I = {p: i for i, p in enumerate(POL)}

# Reference numbers from the locked-split report (validation, 5-seed mean)
REPORT_REF = {'B1_full': (24.65, 35.79), 'T1_future_KL': (25.34, 35.65),
              'Frozen A recognizer (E_A)': (23.21, 33.41)}


def load_tables(annot_csv, split_csv):
    """annotation.csv has no header: 0 clip_id, 1 text, 5 polarity, 6 intensity, 7 emotion, 8 uncertainty."""
    ann = pd.read_csv(annot_csv, header=None, dtype=str).set_index(0)
    sp = pd.read_csv(split_csv, dtype=str)

    def text(c):
        t = ann.at[c, 1] if c in ann.index else None
        return t if isinstance(t, str) else ''

    for k in (1, 2, 3):
        sp[f't{k}'] = sp[f'clip{k}'].map(text)
    sp['yA'] = sp['clip3_emotion'].map(E2I)
    sp['yB'] = sp['clip4_emotion'].map(E2I)
    sp['pA'] = sp['clip3'].map(lambda c: P2I.get(ann.at[c, 5], -1))
    assert sp[['yA', 'yB']].notna().all().all(), 'missing A/B emotion labels'
    return ann, sp


def eval_rows(sp, split, unlock_test=False):
    if split == 'test' and not unlock_test:
        raise RuntimeError('Test split is locked. Set UNLOCK_TEST = True only for the final, preregistered run.')
    return sp[sp['split'] == split].reset_index(drop=True)


def war_uar(pred, y, k):
    pred, y = np.asarray(pred), np.asarray(y)
    war = (pred == y).mean() * 100
    uar = np.mean([(pred[y == c] == c).mean() * 100 for c in range(k) if (y == c).any()])
    return war, uar


def source_boot_ci(pred, y, src, k, n_boot=2000, seed=0):
    """95% CI by resampling whole source folders (episodes) with replacement."""
    pred, y, src = np.asarray(pred), np.asarray(y), np.asarray(src)
    rng = np.random.default_rng(seed)
    groups = [np.where(src == s)[0] for s in np.unique(src)]
    stats = []
    for _ in range(n_boot):
        idx = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])
        stats.append(war_uar(pred[idx], y[idx], k))
    lo, hi = np.percentile(np.array(stats), [2.5, 97.5], axis=0)
    return lo, hi


def report(name, pred, y, src, k=7):
    war, uar = war_uar(pred, y, k)
    lo, hi = source_boot_ci(pred, y, src, k)
    print(f'{name:<46} UAR {uar:5.2f} [{lo[1]:5.1f},{hi[1]:5.1f}]   WAR {war:5.2f} [{lo[0]:5.1f},{hi[0]:5.1f}]')
    return {'name': name, 'UAR': uar, 'WAR': war, 'UAR_lo': lo[1], 'UAR_hi': hi[1], 'WAR_lo': lo[0], 'WAR_hi': hi[0]}


def transition_tables(train_rows, alpha=1.0):
    """P(B | E_A) and P(B | E_A, P_A) estimated on TRAIN gold pairs, add-alpha smoothing."""
    T = np.full((7, 7), alpha)
    TP = np.full((7, 3, 7), alpha)
    for a, p, b in zip(train_rows['yA'], train_rows['pA'], train_rows['yB']):
        T[a, b] += 1
        if p >= 0:
            TP[a, p, b] += 1
    return T / T.sum(1, keepdims=True), TP / TP.sum(2, keepdims=True)


def rtt_forecast(pA_emo, T, pA_pol=None, TP=None):
    """Recognize-then-Transition: B distribution from A posteriors.
    Returns hard (argmax of transition row of argmax A) and soft (expected) B predictions."""
    hard = T[pA_emo.argmax(1)].argmax(1)
    if pA_pol is not None and TP is not None:
        pB = np.einsum('na,np,apb->nb', pA_emo, pA_pol, TP)  # assumes E_A and P_A posteriors independent
    else:
        pB = pA_emo @ T
    return hard, pB.argmax(1), pB

import glob, pickle
import torch, torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ANNOT_CSV = glob.glob(os.path.join(DATASET_DIR, "*", "Hi-EF", "annotation.csv"))[0]
ann, sp = load_tables(ANNOT_CSV, SPLIT_CSV)
sp = sp[sp.split.isin(['train', 'val'])].reset_index(drop=True)      # test rows are dropped here
assert 'test' not in set(sp.split)
if DEBUG_PER_EPISODE:
    sp = sp.groupby('source_folder', group_keys=False).head(DEBUG_PER_EPISODE).reset_index(drop=True)
DEV = sp
train_all = ev = DEV          # names used by the shared feature-loading cell
N = len(DEV)
EPS = np.array(sorted(DEV.source_folder.unique()))
print(f"development MCIS {N} | episodes {len(EPS)}")


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

In [ ]:
# ---- load every clip used by any MCIS (I-IV) once, keep it on the GPU
all_clips = sorted(set(sp[['clip1', 'clip2', 'clip3', 'clip4']].values.ravel()) & set(
    f[:-3].replace('_', '/', 1) for f in os.listdir(FEATURES_DIR) if f.endswith('.pt')))
CIDX = {c: i for i, c in enumerate(all_clips)}
missing = [c for c in set(train_all[['clip1', 'clip2', 'clip3']].values.ravel()) | set(ev[['clip1', 'clip2', 'clip3']].values.ravel())
           if c not in CIDX]
assert not missing, f"{len(missing)} clips without features, e.g. {missing[:3]}"

bufs = {k: [] for k in ('face', 'fmask', 'ori', 'text', 'audio', 'afound')}
for c in tqdm(all_clips, desc='loading features'):
    d = torch.load(os.path.join(FEATURES_DIR, c.replace('/', '_') + '.pt'), map_location='cpu', weights_only=False)
    face = d['face_features'].float()
    fm = d.get('face_valid_mask')
    bufs['face'].append(face)
    bufs['fmask'].append(torch.ones(face.shape[0], dtype=torch.bool) if fm is None else torch.as_tensor(fm).bool().reshape(-1))
    bufs['ori'].append(d['ori_features'].float())
    bufs['text'].append(d['text_feature'].float().reshape(-1))
    bufs['audio'].append(d.get('audio_feature', torch.zeros(527)).float().reshape(-1))
    bufs['afound'].append(torch.tensor(bool(d.get('audio_found', True))))
FEAT = {k: torch.stack(v).to(DEVICE) for k, v in bufs.items()}
del bufs
print({k: tuple(v.shape) for k, v in FEAT.items()})


def gather(idx):
    """idx: LongTensor of clip indices (any shape) -> dict of feature tensors with that leading shape."""
    flat = idx.reshape(-1)
    return {k: v[flat].reshape(*idx.shape, *v.shape[1:]) for k, v in FEAT.items()}

In [ ]:
class TemporalEncoder(nn.Module):
    def __init__(self, d=512, n_frames=16, layers=2, heads=8, dropout=0.1):
        super().__init__()
        self.pos = nn.Parameter(torch.randn(1, n_frames, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, heads, 4 * d, dropout, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)

    def forward(self, x, mask):  # mask: True = valid frame
        mask = mask.clone()
        mask[~mask.any(1), 0] = True
        h = self.enc(x + self.pos[:, :x.size(1)], src_key_padding_mask=~mask)
        m = mask.unsqueeze(-1).float()
        return (h * m).sum(1) / m.sum(1)


class ClipEncoder(nn.Module):
    """Face/original temporal encoders + text/audio tokens -> 1-layer fusion Transformer -> one 512-d vector."""

    def __init__(self, d=512):
        super().__init__()
        self.face, self.ori = TemporalEncoder(d), TemporalEncoder(d)
        self.text = nn.Sequential(nn.LayerNorm(512), nn.Linear(512, d))
        self.audio = nn.Sequential(nn.LayerNorm(527), nn.Linear(527, d))
        self.modality = nn.Parameter(torch.randn(1, 4, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, 8, 4 * d, 0.1, batch_first=True, norm_first=True)
        self.fusion = nn.TransformerEncoder(layer, 1, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)

    def forward(self, b):
        ori_mask = torch.ones(b['ori'].shape[:2], dtype=torch.bool, device=b['ori'].device)
        tokens = torch.stack([self.face(b['face'], b['fmask']), self.ori(b['ori'], ori_mask),
                              self.text(b['text']), self.audio(F.normalize(b['audio'], dim=-1))], 1)
        valid = torch.ones(tokens.shape[:2], dtype=torch.bool, device=tokens.device)
        valid[:, 3] = b['afound']
        h = self.fusion(tokens + self.modality, src_key_padding_mask=~valid)
        m = valid.unsqueeze(-1).float()
        return self.norm((h * m).sum(1) / m.sum(1))


class ClipRecognizer(nn.Module):
    def __init__(self, d=512):
        super().__init__()
        self.enc = ClipEncoder(d)
        self.drop = nn.Dropout(0.3)
        self.emo, self.pol = nn.Linear(d, 7), nn.Linear(d, 3)

    def forward(self, b):
        h = self.drop(self.enc(b))
        return self.emo(h), self.pol(h)


N_REC = 12   # 7 emotion probs + 3 polarity probs + max prob + entropy


class Forecaster(nn.Module):
    def __init__(self, use_raw=True, use_traj=False, d=512, positions=None):
        super().__init__()
        self.use_raw, self.use_traj = use_raw, use_traj
        self.positions = positions   # clip positions (0=I, 1=II, 2=III); None = the last n clips
        self.enc = ClipEncoder(d) if use_raw else None
        self.traj = nn.Sequential(nn.LayerNorm(N_REC), nn.Linear(N_REC, d), nn.GELU(), nn.Linear(d, d)) if use_traj else None
        self.clip_pos = nn.Parameter(torch.randn(1, 3, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, 8, 4 * d, 0.1, batch_first=True, norm_first=True)
        self.inter = nn.TransformerEncoder(layer, 2, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.3), nn.Linear(d, d // 2), nn.GELU(),
                                  nn.Dropout(0.2), nn.Linear(d // 2, 7))

    def forward(self, clip_idx, rec):  # clip_idx [B,n], rec [B,n,N_REC], n <= 3 clips in temporal order
        B, n = clip_idx.shape
        tok = 0
        if self.use_raw:
            feats = gather(clip_idx)
            flat = {k: v.reshape(B * n, *v.shape[2:]) for k, v in feats.items()}
            tok = self.enc(flat).reshape(B, n, -1)
        if self.use_traj:
            tok = tok + self.traj(rec)
        pos = self.clip_pos[:, list(self.positions)] if self.positions is not None else self.clip_pos[:, 3 - n:]
        h = self.inter(tok + pos)
        return self.head(h.mean(1))

## From G8a features to role-tagged slots

In [ ]:
from collections import defaultdict
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA

G8 = {}
for f in sorted(glob.glob(os.path.join(G8A_DIR, '**', 'shard_*.pkl'), recursive=True)):
    G8.update(pickle.load(open(f, 'rb')))
need = sorted(set(DEV[['clip1', 'clip2', 'clip3']].values.ravel()))
miss = [c for c in need if c not in G8]
assert not miss, f"{len(miss)} clips missing from G8a, e.g. {miss[:3]}"


def softmax(z):
    e = np.exp(z - z.max(-1, keepdims=True))
    return e / e.sum(-1, keepdims=True)


# per-clip frame features except the PCA part (18 dims), and the raw HSEmotion embeddings
FB, EMB = {}, {}
for c in need:
    fs = G8[c]['faces']
    if not fs:
        FB[c], EMB[c] = np.zeros((0, 18), np.float32), None
        continue
    dur = max(G8[c]['meta'].get('duration') or 0.0, 1e-3)
    fer = np.stack([d['fer'] for d in fs]).astype(np.float32)
    box = np.stack([d['box'] for d in fs])
    FB[c] = np.concatenate([
        softmax(fer[:, :8]), fer[:, 8:10], np.stack([d['pose'] for d in fs]) / 90.0,
        np.stack([(box[:, 0] + box[:, 2]) / 2, (box[:, 1] + box[:, 3]) / 2,
                  np.sqrt(np.clip((box[:, 2] - box[:, 0]) * (box[:, 3] - box[:, 1]), 0, None))], 1),
        np.nan_to_num(np.array([[d['mouth'] * 10] for d in fs], np.float32)),
        np.array([[min(d['t'] / dur, 1.0)] for d in fs], np.float32)], 1).astype(np.float32)
    EMB[c] = np.stack([d['fer_emb'] for d in fs]) if all('fer_emb' in d for d in fs) else None
HAS_EMB = all(EMB[c] is not None for c in need if len(G8[c]['faces']))
FDIM = (PCA_DIM if HAS_EMB else 0) + 18
print(f"HSEmotion embedding available: {HAS_EMB} | frame feature dim {FDIM}")


def pick_frames(idx, cap):
    return idx if len(idx) <= cap else [idx[i] for i in np.linspace(0, len(idx) - 1, cap).astype(int)]


def sync(c, face_ids):
    # correlation of mouth opening with the audio energy envelope, and mouth variability, for a set of faces
    A = G8[c]['audio']
    fs = [G8[c]['faces'][j] for j in face_ids]
    fs = [d for d in fs if np.isfinite(d['mouth'])]
    if A is None or len(fs) < 4:
        return 0.0, 0.0
    m = np.array([d['mouth'] for d in fs])
    env = A['env']
    e = np.array([env[min(int(d['t'] * 10), len(env) - 1)] for d in fs]) if len(env) else np.zeros(len(fs))
    r = float(np.corrcoef(m, e)[0, 1]) if m.std() > 1e-6 and e.std() > 1e-9 else 0.0
    return r, float(m.std() * 10)


def mean12(c_faces, n_sampled):
    if not c_faces:
        return np.zeros(12, np.float32)
    fer = np.stack([d['fer'] for d in c_faces]).astype(np.float32)
    v = np.concatenate([softmax(fer[:, :8]), fer[:, 8:10]], 1).mean(0)
    return np.concatenate([v, [1.0, len({d['frame'] for d in c_faces}) / max(n_sampled, 1)]]).astype(np.float32)


NVOICE = 3 * 2 + 3
SLOT, PSLOT = {}, {}                      # (n, role, clip) / (n, clip) -> (clip id, face indices)
FMASK = np.zeros((N, 3, 3, MAXF), bool); PMASK = np.zeros((N, 1, 3, MAXF_POOL), bool)
VOI = np.zeros((N, 3, NVOICE), np.float32)
LRF = np.zeros((N, 60), np.float32)       # G6b-style role means (late fusion and the joint-branch arm)
CENT_COS = np.full(N, np.nan, np.float32)
for n, row in enumerate(tqdm(DEV.itertuples(), total=N, desc='roles')):
    cl = [row.clip1, row.clip2, row.clip3]
    items = [(k, j) for k, c in enumerate(cl) for j in range(len(G8[c]['faces']))]
    lab = np.zeros(len(items), int)
    E = np.stack([G8[cl[k]]['faces'][j]['arc'] for k, j in items]).astype(np.float32) if items else None
    if len(items) > 1:
        lab = AgglomerativeClustering(n_clusters=None, metric='cosine', linkage='average',
                                      distance_threshold=1 - SAME_PERSON_COS).fit_predict(E)
    frames = defaultdict(set)
    for (k, j), p in zip(items, lab):
        frames[(k, p)].add(G8[cl[k]]['faces'][j]['frame'])
    ids3 = sorted({p for (k, p) in frames if k == 2}, key=lambda p: -len(frames[(2, p)]))
    A = ids3[0] if ids3 else None
    L = ids3[1] if len(ids3) > 1 else None
    if A is not None and L is not None:
        ca, cb = E[lab == A].mean(0), E[lab == L].mean(0)
        CENT_COS[n] = float(ca @ cb / (np.linalg.norm(ca) * np.linalg.norm(cb) + 1e-9))
    role = lambda p: 0 if p == A else (1 if p == L else 2)
    by = defaultdict(list)
    for (k, j), p in zip(items, lab):
        by[(role(p), k)].append(j)
        by[('pool', k)].append(j)
    for k, c in enumerate(cl):
        for r in range(3):
            idx = pick_frames(sorted(by[(r, k)], key=lambda j: G8[c]['faces'][j]['t']), MAXF)
            SLOT[(n, r, k)] = (c, idx); FMASK[n, r, k, :len(idx)] = True
        idx = pick_frames(sorted(by[('pool', k)], key=lambda j: G8[c]['faces'][j]['t']), MAXF_POOL)
        PSLOT[(n, k)] = (c, idx); PMASK[n, 0, k, :len(idx)] = True
        v = [x for r in range(3) for x in sync(c, by[(r, k)])]
        a3, ak = G8[cl[2]]['audio'], G8[c]['audio']
        ok = a3 is not None and ak is not None and a3.get('ecapa') is not None and ak.get('ecapa') is not None
        vcos = float(a3['ecapa'].astype(np.float32) @ ak['ecapa'].astype(np.float32)) if ok else 0.0
        VOI[n, k] = v + [vcos, float(ok), len(set(lab)) / 5.0]

    def faces_of(k, p):
        return [G8[cl[k]]['faces'][j] for (kk, j), q in zip(items, lab) if kk == k and q == p]

    def dominant(k):
        ns = G8[cl[k]]['meta']['n_sampled']
        cand = sorted({q for (kk, q) in frames if kk == k}, key=lambda q: -len(frames[(k, q)]))
        return cand[0] if cand and len(frames[(k, cand[0])]) / max(ns, 1) >= DOMINANT_MIN_FRAC else None

    n3 = G8[cl[2]]['meta']['n_sampled']
    blocks = [mean12(faces_of(2, A) if A is not None else [], n3), mean12(faces_of(2, L) if L is not None else [], n3),
              mean12((faces_of(0, L) + faces_of(1, L)) if L is not None else [], n3)]
    for k in (1, 0):
        d = dominant(k)
        blocks.append(mean12(faces_of(k, d) if d is not None else [], G8[cl[k]]['meta']['n_sampled']))
    LRF[n] = np.concatenate(blocks)

VIS = FMASK[:, 1, 2].any(-1)
print(f"A found in III {FMASK[:, 0, 2].any(-1).mean() * 100:.1f}% | listener visible in III {VIS.mean() * 100:.1f}% | "
      f"listener also in I/II {FMASK[:, 1, :2].any((-1, -2)).mean() * 100:.1f}%")


def build_face_tensors(fit_clips):
    # PCA of the HSEmotion embedding fitted on faces of the training clips of the current fold only
    pca, var = None, None
    if HAS_EMB:
        pool = np.concatenate([EMB[c] for c in fit_clips if EMB.get(c) is not None])
        pick = np.random.default_rng(0).choice(len(pool), min(60000, len(pool)), replace=False)
        pca = PCA(PCA_DIM, random_state=0).fit(pool[pick].astype(np.float32))
        var = float(pca.explained_variance_ratio_.sum())
    FV = {}
    for c in need:
        if len(FB[c]) == 0:
            FV[c] = np.zeros((0, FDIM), np.float32)
        else:
            FV[c] = np.concatenate([pca.transform(EMB[c].astype(np.float32)), FB[c]], 1) if HAS_EMB else FB[c]
    Fa = np.zeros((N, 3, 3, MAXF, FDIM), np.float16)
    Pa = np.zeros((N, 1, 3, MAXF_POOL, FDIM), np.float16)
    for (n, r, k), (c, idx) in SLOT.items():
        if idx:
            Fa[n, r, k, :len(idx)] = FV[c][idx]
    for (n, k), (c, idx) in PSLOT.items():
        if idx:
            Pa[n, 0, k, :len(idx)] = FV[c][idx]
    return torch.tensor(Fa, device=DEVICE), torch.tensor(Pa, device=DEVICE), var

## Models

In [ ]:
T = lambda a, dt=None: torch.tensor(a, device=DEVICE) if dt is None else torch.tensor(a, dtype=dt, device=DEVICE)
FMASK, PMASK, VOI = T(FMASK), T(PMASK), T(VOI)
FACE = POOL = LRFZ = None             # set per fold
CLIPIDX = T([[CIDX[c] for c in r] for r in DEV[['clip1', 'clip2', 'clip3']].values])
TXT, AUD, AFD = FEAT['text'][CLIPIDX], F.normalize(FEAT['audio'][CLIPIDX], dim=-1), FEAT['afound'][CLIPIDX].float()
fm = FEAT['fmask'][CLIPIDX].unsqueeze(-1).float()
SCN = torch.cat([FEAT['ori'][CLIPIDX].mean(2), (FEAT['face'][CLIPIDX] * fm).sum(2) / fm.sum(2).clamp(min=1)], -1)
ZREC = torch.zeros(N, 3, N_REC, device=DEVICE)
YB, YA = T(DEV.yB.values), T(DEV.yA.values)


class FramePool(nn.Module):
    def __init__(self, fin, d):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(fin), nn.Linear(fin, d), nn.GELU(), nn.Linear(d, d))
        self.score = nn.Linear(d, 1)

    def forward(self, x, m):                      # x [..., F, fin], m [..., F]
        h = self.proj(x.float())
        a = self.score(h).squeeze(-1).masked_fill(~m, -1e4)
        w = torch.softmax(a, -1) * m.float()
        return (w.unsqueeze(-1) * h).sum(-2), m.any(-1)


class RoleNet(nn.Module):
    def __init__(self, cfg, d=RN['D']):
        super().__init__()
        self.cfg, self.R = cfg, (3 if cfg['role'] else 1)
        if cfg['faces']:
            self.pool = FramePool(FDIM, d)
            self.absent = nn.Parameter(torch.randn(self.R, 3, d) * 0.02)
            self.face_role = nn.Parameter(torch.randn(self.R, d) * 0.02)
        if cfg['ctx']:
            self.text = nn.Sequential(nn.LayerNorm(512), nn.Linear(512, d))
            self.audio = nn.Sequential(nn.LayerNorm(527), nn.Linear(527, d))
            self.voice = nn.Linear(NVOICE, d)
            self.scene = nn.Sequential(nn.LayerNorm(1024), nn.Linear(1024, d))
            self.ctx_role = nn.Parameter(torch.randn(2, d) * 0.02)
        self.clip_emb = nn.Parameter(torch.randn(3, d) * 0.02)
        self.query = nn.Parameter(torch.randn(1, 1, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, RN['heads'], 4 * d, RN['dropout'], batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, RN['layers'], enable_nested_tensor=False)
        mk = lambda: nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.3), nn.Linear(d, 7))
        self.head = mk()
        self.head_face = mk() if cfg['aux'] and cfg['faces'] else None
        self.head_ctx = mk() if cfg['aux'] and cfg['ctx'] else None
        self.head_A = mk() if cfg['aux'] and cfg['faces'] and cfg['role'] else None

    def forward(self, ix, train=False):
        B, groups, aux = len(ix), [], {}
        if self.cfg['faces']:
            x, m = (FACE[ix], FMASK[ix]) if self.R == 3 else (POOL[ix], PMASK[ix])
            h, present = self.pool(x, m)                                      # [B, R, 3, d]
            h = torch.where(present.unsqueeze(-1), h, self.absent.unsqueeze(0).expand(B, -1, -1, -1))
            h = h + self.face_role[None, :, None] + self.clip_emb[None, None]
            ft = h.reshape(B, self.R * 3, -1)
            groups.append(ft)
            if self.head_face is not None:
                aux['face'] = (self.head_face(ft.mean(1)), YB[ix], RN['aux_w'])
            if self.head_A is not None:
                tA = torch.where(present[:, 0, 2], YA[ix], torch.full_like(YA[ix], -100))
                aux['A'] = (self.head_A(h[:, 0, 2]), tA, RN['a_w'])
        if self.cfg['ctx']:
            spk_ = self.text(TXT[ix]) + self.audio(AUD[ix]) * AFD[ix].unsqueeze(-1) + self.voice(VOI[ix]) + self.ctx_role[0]
            scn = self.scene(SCN[ix]) + self.ctx_role[1]
            ct = torch.cat([spk_ + self.clip_emb, scn + self.clip_emb], 1)     # [B, 6, d]
            groups.append(ct)
            if self.head_ctx is not None:
                aux['ctx'] = (self.head_ctx(ct.mean(1)), YB[ix], RN['aux_w'])
        toks = torch.cat([self.query.expand(B, -1, -1)] + groups, 1)
        valid = torch.ones(toks.shape[:2], dtype=torch.bool, device=toks.device)
        if train and self.cfg['mdrop'] and len(groups) == 2:
            u = torch.rand(B, device=toks.device)
            drop_ctx = u < RN['p_drop_ctx']
            drop_face = (u >= RN['p_drop_ctx']) & (u < RN['p_drop_ctx'] + RN['p_drop_face'])
            nf = groups[0].shape[1]
            valid[:, 1:1 + nf] &= ~drop_face.unsqueeze(1)
            valid[:, 1 + nf:] &= ~drop_ctx.unsqueeze(1)
        out = self.enc(toks, src_key_padding_mask=~valid)
        return self.head(out[:, 0]), aux


class B1Wrap(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = Forecaster(use_raw=True, use_traj=False)

    def forward(self, ix, train=False):
        return self.f(CLIPIDX[ix], ZREC[ix]), {}


class B1FaceWrap(nn.Module):
    # B1 + the 60-d G6b-style role-face vector through a zero-initialised branch, trained jointly (the G7b design)
    def __init__(self, n_face=60, d=512):
        super().__init__()
        self.f = Forecaster(use_raw=True, use_traj=False, d=d)
        self.face = nn.Sequential(nn.Linear(n_face, d // 2), nn.GELU(), nn.Dropout(0.3), nn.Linear(d // 2, d))
        nn.init.zeros_(self.face[-1].weight); nn.init.zeros_(self.face[-1].bias)

    def forward(self, ix, train=False):
        b, clip_idx = self.f, CLIPIDX[ix]
        B, n = clip_idx.shape
        feats = gather(clip_idx)
        tok = b.enc({k: v.reshape(B * n, *v.shape[2:]) for k, v in feats.items()}).reshape(B, n, -1)
        h = b.inter(tok + b.clip_pos[:, 3 - n:])
        return b.head(h.mean(1) + self.face(LRFZ[ix])), {}


HP = {'b1': dict(lr=LR, wd=WEIGHT_DECAY, epochs=FC_EPOCHS, patience=PATIENCE, batch=FC_BATCH),
      'role': dict(lr=RN['lr'], wd=RN['wd'], epochs=RN['epochs'], patience=RN['patience'], batch=RN['batch'])}
HP['b1face'] = HP['b1']
MAKE = {'b1': lambda cfg: B1Wrap(), 'b1face': lambda cfg: B1FaceWrap(), 'role': lambda cfg: RoleNet(cfg)}
print("parameters:", {n: f"{sum(p.numel() for p in MAKE[k](c).parameters()) / 1e6:.2f}M" for n, k, c in EXPERIMENTS})


def predict(model, ix, bs=256):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(ix), bs):
            out.append(F.softmax(model(ix[i:i + bs])[0], -1).cpu())
    return torch.cat(out).numpy()


def train_eval(kind, cfg, tr, dev, te, seed):
    seed_all(seed)
    hp = HP[kind]
    model = MAKE[kind](cfg).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=hp['lr'], weight_decay=hp['wd'])
    y_dev = YB[dev].cpu().numpy()
    best, best_state, bad = -1, None, 0
    for ep in range(hp['epochs']):
        model.train()
        perm = tr[torch.randperm(len(tr), device=DEVICE)]
        for i in range(0, len(perm), hp['batch']):
            j = perm[i:i + hp['batch']]
            logits, aux = model(j, train=True)
            loss = F.cross_entropy(logits, YB[j])
            for l, t, w in aux.values():
                if (t >= 0).any():
                    loss = loss + w * F.cross_entropy(l, t, ignore_index=-100)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        u = war_uar(predict(model, dev).argmax(1), y_dev, 7)[1]
        if u > best:
            best, bad = u, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= hp['patience']:
                break
    model.load_state_dict(best_state)
    return predict(model, te), best

## RoleNet with token-level switches

In [ ]:
class RoleNetTok(RoleNet):
    # RoleNet with switches for the readout, the embeddings, the absent token, and which speech/scene/clip tokens exist.
    # Token layout: [queries] + 9 face tokens (role r, clip k at 3r+k) + 3 speech tokens (clip k) + 3 scene tokens (clip k).
    def __init__(self, cfg, d=RN['D']):
        super().__init__(FULL, d)
        self.t = cfg
        self.nq = cfg['nq'] if cfg['readout'] == 'query' else 0
        if self.nq > 1:
            self.queries = nn.Parameter(torch.randn(1, self.nq, d) * 0.02)
        keep_clip = torch.zeros(3, dtype=torch.bool, device=DEVICE)
        keep_clip[list(cfg['clips'])] = True
        kf = keep_clip.repeat(3)                                              # face token 3r+k -> clip k
        if 'L' in cfg['drop']:
            kf[3:6] = False                                                   # listener tokens, all clips (G11 minus-L)
        ks, kc = keep_clip.clone(), keep_clip.clone()
        if 'speech' in cfg['drop']:
            ks[:] = False
        if 'speechIII' in cfg['drop']:
            ks[2] = False
        if 'scene' in cfg['drop']:
            kc[:] = False
        self.keep_face, self.keep_ctx = kf, torch.cat([ks, kc])
        if not bool(self.keep_ctx.any()):
            self.head_ctx = None
        if not bool(kf[2]):                                                  # A's clip-III token gone
            self.head_A = None

    def forward(self, ix, train=False):
        B, t = len(ix), self.t
        d = self.query.shape[-1]
        zero = torch.zeros(1, device=DEVICE)
        role_e = self.face_role if t['role_emb'] else torch.zeros_like(self.face_role)
        clip_e = self.clip_emb if t['clip_emb'] else torch.zeros_like(self.clip_emb)
        aux = {}
        h, present = self.pool(FACE[ix], FMASK[ix])                           # [B, 3, 3, d], [B, 3, 3]
        if t['absent'] == 'token':
            h = torch.where(present.unsqueeze(-1), h, self.absent.unsqueeze(0).expand(B, -1, -1, -1))
            vf = self.keep_face.unsqueeze(0).expand(B, -1)
        else:
            vf = self.keep_face.unsqueeze(0) & present.reshape(B, 9)
        h = h + role_e[None, :, None] + clip_e[None, None]
        ft = h.reshape(B, 9, -1)
        parts = {'text': self.text(TXT[ix]), 'audio': self.audio(AUD[ix]) * AFD[ix].unsqueeze(-1), 'voice': self.voice(VOI[ix])}
        spk_ = sum(parts[p] for p in t['speech_parts']) + self.ctx_role[0] + clip_e
        scn = self.scene(SCN[ix]) + self.ctx_role[1] + clip_e
        ct = torch.cat([spk_, scn], 1)                                        # [B, 6, d]
        vc = self.keep_ctx.unsqueeze(0).expand(B, -1)
        if self.head_face is not None:
            w = vf.float().unsqueeze(-1)
            aux['face'] = (self.head_face((ft * w).sum(1) / w.sum(1).clamp(min=1)), YB[ix], RN['aux_w'])
        if self.head_A is not None:
            tA = torch.where(present[:, 0, 2], YA[ix], torch.full_like(YA[ix], -100))
            aux['A'] = (self.head_A(h[:, 0, 2]), tA, RN['a_w'])
        if self.head_ctx is not None:
            w = vc.float().unsqueeze(-1)
            aux['ctx'] = (self.head_ctx((ct * w).sum(1) / w.sum(1).clamp(min=1)), YB[ix], RN['aux_w'])
        vf, vc = vf.clone(), vc.clone()
        if train and self.cfg['mdrop']:
            u = torch.rand(B, device=DEVICE)
            drop_ctx = u < RN['p_drop_ctx']
            drop_face = (u >= RN['p_drop_ctx']) & (u < RN['p_drop_ctx'] + RN['p_drop_face'])
            vf &= ~drop_face.unsqueeze(1)
            vc &= ~drop_ctx.unsqueeze(1)
        nq = self.nq
        q = [] if nq == 0 else [(self.query if nq == 1 else self.queries).expand(B, -1, -1)]
        toks = torch.cat(q + [ft, ct], 1)
        valid = torch.cat([torch.ones(B, nq, dtype=torch.bool, device=DEVICE), vf, vc], 1)
        # never leave a sample without a valid data token (an all-masked row would produce NaN through attention)
        valid[:, nq] |= ~valid[:, nq:].any(1)
        mask = None
        if nq and t['q_readonly']:
            T_ = toks.shape[1]
            mask = torch.zeros(T_, T_, dtype=torch.bool, device=DEVICE)
            mask[nq:, :nq] = True                                            # data tokens cannot attend to the query
        out = self.enc(toks, mask=mask, src_key_padding_mask=~valid)
        if t['readout'] == 'query':
            z = out[:, :nq].mean(1)
        elif t['readout'] == 'mean':
            w = valid.float().unsqueeze(-1)
            z = (out * w).sum(1) / w.sum(1).clamp(min=1)
        else:                                                                 # 'L3': listener face token of clip III
            z = out[:, nq + 3 * 1 + 2]
        return self.head(z), aux


MAKE['tok'] = lambda cfg: RoleNetTok(cfg)
HP['tok'] = HP['role']
EXPERIMENTS = ARMS
print("parameters:", {n: f"{sum(p.numel() for p in MAKE[k](c).parameters()) / 1e6:.3f}M" for n, k, c in EXPERIMENTS})

## G20 models: additive, local interaction, single-source branches; EMAP

In [ ]:
from scipy.optimize import minimize_scalar


class XBranch(nn.Module):
    # text only: query + 3 text tokens (one per clip) -> the RoleNet Transformer -> 7 logits
    def __init__(self, d=RN['D']):
        super().__init__()
        self.text = nn.Sequential(nn.LayerNorm(512), nn.Linear(512, d))
        self.clip_emb = nn.Parameter(torch.randn(3, d) * 0.02)
        self.query = nn.Parameter(torch.randn(1, 1, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, RN['heads'], 4 * d, RN['dropout'], batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, RN['layers'], enable_nested_tensor=False)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(0.3), nn.Linear(d, 7))

    def forward(self, ix, train=False):
        toks = torch.cat([self.query.expand(len(ix), -1, -1), self.text(TXT[ix]) + self.clip_emb], 1)
        return self.head(self.enc(toks)[:, 0]), {}


class LocalR(nn.Module):
    # r_y(x_t, a_t) = W_y (P x_t ⊙ Q a_t): pure bilinear text x audio term of one clip, shared over clips.
    # No affine LayerNorm and no biases, so r contains no additive part; zero at init; 0 for clips without audio.
    def __init__(self, m=R_DIM):
        super().__init__()
        self.px = nn.Sequential(nn.LayerNorm(512, elementwise_affine=False), nn.Linear(512, m, bias=False))
        self.pa = nn.Sequential(nn.LayerNorm(527, elementwise_affine=False), nn.Linear(527, m, bias=False))
        self.drop, self.out = nn.Dropout(0.3), nn.Linear(m, 7, bias=False)
        nn.init.zeros_(self.out.weight)

    def forward(self, ix):
        r = self.out(self.drop(self.px(TXT[ix]) * self.pa(AUD[ix])))              # [B, 3, 7]
        return (r * AFD[ix].unsqueeze(-1)).sum(1)


class AdditiveNet(nn.Module):
    # a_y(X) + b_y(Z) [+ sum_t r_y(x_t, a_t)]; the branches exchange no information and are trained jointly on the sum
    def __init__(self, local=False):
        super().__init__()
        self.bx, self.bz = XBranch(), RoleNetTok(ZCFG)
        self.r = LocalR() if local else None

    def forward(self, ix, train=False):
        lx, _ = self.bx(ix, train)
        lz, aux = self.bz(ix, train)                                              # aux heads of the Z branch see Z only
        l = lx + lz
        if self.r is not None:
            l = l + self.r(ix)
        return l, aux


MAKE.update({'add': lambda cfg: AdditiveNet(False), 'local': lambda cfg: AdditiveNet(True),
             'xonly': lambda cfg: XBranch(), 'zonly': lambda cfg: RoleNetTok(ZCFG)})
for k in ('add', 'local', 'xonly', 'zonly'):
    HP[k] = HP['role']
G20_ARMS = [("Additive", 'add', None), ("Local", 'local', None), ("Window", 'tok', BASE),
            ("X-only", 'xonly', None), ("Z-only", 'zonly', None)]
print("parameters:", {n: f"{sum(p.numel() for p in MAKE[k](c).parameters() if p.requires_grad) / 1e6:.3f}M"
                      for n, k, c in G20_ARMS})


def fit_model(kind, cfg, tr, dev, seed):
    # identical to train_eval (optimiser, aux losses, early stopping on dev UAR) but returns the trained model
    seed_all(seed)
    hp = HP[kind]
    model = MAKE[kind](cfg).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=hp['lr'], weight_decay=hp['wd'])
    y_dev = YB[dev].cpu().numpy()
    best, best_state, bad = -1, None, 0
    for ep in range(hp['epochs']):
        model.train()
        perm = tr[torch.randperm(len(tr), device=DEVICE)]
        for i in range(0, len(perm), hp['batch']):
            j = perm[i:i + hp['batch']]
            logits, aux = model(j, train=True)
            loss = F.cross_entropy(logits, YB[j])
            for l, t, w in aux.values():
                if (t >= 0).any():
                    loss = loss + w * F.cross_entropy(l, t, ignore_index=-100)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        u = war_uar(predict(model, dev).argmax(1), y_dev, 7)[1]
        if u > best:
            best, bad = u, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= hp['patience']:
                break
    model.load_state_dict(best_state)
    model.eval()
    return model, best


def logits_of(model, ix, bs=512):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(ix[i:i + bs])[0] for i in range(0, len(ix), bs)]).cpu().numpy().astype(np.float64)


def emap(model, ix, bs=512):
    # EMAP on logits: mean_j f(x_i, z_j) + mean_j f(x_j, z_i) - mean_jk f(x_j, z_k) over the cross-pairs of ix.
    # X (text) is swapped by pointing the global TXT to a copy whose rows ix carry the text of rows ix[(a + s) % n].
    global TXT
    n = len(ix)
    Mx, Mz = torch.zeros(n, 7, device=DEVICE), torch.zeros(n, 7, device=DEVICE)
    TXT0, TXT2 = TXT, TXT.clone()
    model.eval()
    try:
        TXT = TXT2
        with torch.no_grad():
            for s in range(n):
                sh = (torch.arange(n, device=DEVICE) + s) % n
                TXT2[ix] = TXT0[ix[sh]]
                L = torch.cat([model(ix[i:i + bs])[0] for i in range(0, n, bs)])   # row a: f(x_{sh[a]}, z_a)
                Mz += L
                Mx.index_add_(0, sh, L)
    finally:
        TXT = TXT0
    tot = Mz.sum(0, keepdim=True) / n ** 2
    return ((Mx + Mz) / n - tot).cpu().numpy().astype(np.float64)


def softmax_np(L):
    L = L - L.max(1, keepdims=True)
    E_ = np.exp(L)
    return E_ / E_.sum(1, keepdims=True)


def nll(P, y):
    return float(-np.log(P[np.arange(len(y)), y] + 1e-12).mean())


def fit_T(L, y):
    return minimize_scalar(lambda t: nll(softmax_np(L / t), y), bounds=(0.05, 20), method='bounded').x


def log_softmax_np(L):
    L = L - L.max(1, keepdims=True)
    return L - np.log(np.exp(L).sum(1, keepdims=True))

## Neural family: 5-fold episode CV (same folds, early-stopping episodes and seeds as G8b–G19)

In [ ]:
import re

sizes = DEV.source_folder.value_counts()
order = list(sizes.index)
random.Random(0).shuffle(order)
order = sorted(order, key=lambda e: -sizes[e])
load_, FOLD = [0] * N_OUTER, {}
for e in order:
    f = int(np.argmin(load_)); FOLD[e] = f; load_[f] += sizes[e]
fold_of_row = DEV.source_folder.map(FOLD).values
print("fold sizes (MCIS):", load_)

y_all = DEV.yB.values
src = DEV.source_folder.values
NAMES = [a[0] for a in G20_ARMS] + ['LateFusion', 'Window-EMAP']
OOF = {k: np.full((len(SEEDS), N, 7), np.nan, np.float32) for k in NAMES}     # calibrated probabilities
log, temps = [], []
t0 = time.time()
for f in range(N_OUTER):
    tr_eps = [e for e in EPS if FOLD[e] != f]
    dev_eps = sorted(random.Random(100 + f).sample(tr_eps, N_INNER_DEV))
    trr = np.where(np.isin(src, tr_eps))[0]
    fit_rows = np.where(np.isin(src, tr_eps) & ~np.isin(src, dev_eps))[0]
    dev_rows = np.where(np.isin(src, dev_eps))[0]
    te_rows = np.where(fold_of_row == f)[0]
    fit_clips = sorted(set(DEV.iloc[trr][['clip1', 'clip2', 'clip3']].values.ravel()))
    FACE, POOL, var = build_face_tensors(fit_clips)
    print(f"fold {f}: train {len(fit_rows)} | early-stop {len(dev_rows)} | eval {len(te_rows)}", flush=True)
    tr, dev, te = T(fit_rows), T(dev_rows), T(te_rows)
    y_dev, y_te = y_all[dev_rows], y_all[te_rows]
    for si, seed in enumerate(SEEDS):
        LG = {}
        for name, kind, cfg in G20_ARMS:
            model, sel = fit_model(kind, cfg, tr, dev, seed + 1000 * f)
            LG[name] = (logits_of(model, dev), logits_of(model, te))
            if name == 'Window':
                LG['Window-EMAP'] = (emap(model, dev), emap(model, te))
            if name == 'Additive' and si == 0:
                err = np.abs(emap(model, te) - LG[name][1]).max()
                print(f"  EMAP sanity check on the additive model (should be ~0): max |EMAP - f| = {err:.2e}")
                assert err < 1e-2, "EMAP does not reproduce an additive model"
            log.append({'fold': f, 'seed': seed, 'arm': name, 'sel_UAR': sel})
            del model
            torch.cuda.empty_cache()
        LG['LateFusion'] = tuple(log_softmax_np(LG['X-only'][i]) + log_softmax_np(LG['Z-only'][i]) for i in (0, 1))
        msg = []
        for k in NAMES:
            Ld, Lt = LG[k]
            t_ = fit_T(Ld, y_dev)
            OOF[k][si, te_rows] = softmax_np(Lt / t_)
            temps.append({'fold': f, 'seed': seed, 'arm': k, 'T': t_})
            msg.append(f"{k} {nll(OOF[k][si, te_rows], y_te):.3f}")
        print(f"fold {f} seed {seed}: NLL " + " | ".join(msg) + f" | {(time.time() - t0) / 60:.1f} min", flush=True)

assert all(not np.isnan(v).any() for v in OOF.values())
safe = lambda s: re.sub(r'[^0-9A-Za-z]+', '_', s).strip('_')
pd.DataFrame(log).to_csv(f"{OUT_DIR}/g20_fold_seed_log.csv", index=False)
pd.DataFrame(temps).to_csv(f"{OUT_DIR}/g20_temperatures.csv", index=False)
np.savez(f"{OUT_DIR}/g20_oof_probs.npz", sample_id=DEV.sample_id.values, fold=fold_of_row, y=y_all, src=src,
         **{safe(k): v for k, v in OOF.items()})
print("saved g20_oof_probs.npz, g20_fold_seed_log.csv, g20_temperatures.csv")

## Logistic family (same outer folds; C and temperature chosen inside each training part)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA as _PCA

EXPR = np.zeros((N, 3, 11), np.float32)
for n, row in enumerate(DEV.itertuples()):
    for k, c in enumerate((row.clip1, row.clip2, row.clip3)):
        if len(FB[c]):
            EXPR[n, k, :10] = FB[c][:, :10].mean(0); EXPR[n, k, 10] = 1.0
TXTn, AUDn, SCNn = TXT.cpu().numpy(), AUD.cpu().numpy(), SCN.cpu().numpy()
VOIn, AFDn = VOI.cpu().numpy(), AFD.cpu().numpy()


def pca_scores(M, tr, te, k, fit_mask=None):
    rows = tr if fit_mask is None else tr[fit_mask[tr]]
    p = _PCA(min(k, len(rows) - 1, M.shape[1]), random_state=0).fit(M[rows])
    return p.transform(M[tr]), p.transform(M[te])


def design(tr, te):
    # all transforms fitted on rows tr only; returns {block: (train matrix, eval matrix)}
    X_tr, X_te, Z_tr, Z_te = [], [], [], []
    for k in range(3):
        a, b = pca_scores(TXTn[:, k], tr, te, PCA_K); X_tr.append(a); X_te.append(b)
        a, b = pca_scores(AUDn[:, k], tr, te, PCA_K, AFDn[:, k] > 0)
        Z_tr.append(a * AFDn[tr, k:k + 1]); Z_te.append(b * AFDn[te, k:k + 1])
        a, b = pca_scores(SCNn[:, k], tr, te, PCA_K); Z_tr.append(a); Z_te.append(b)
        for M in (EXPR[:, k], VOIn[:, k], AFDn[:, k:k + 1]):
            Z_tr.append(M[tr]); Z_te.append(M[te])
    D = {'X': (np.concatenate(X_tr, 1), np.concatenate(X_te, 1)), 'Z': (np.concatenate(Z_tr, 1), np.concatenate(Z_te, 1))}
    # local: sum_t vec(x~_t a~_t^T), x~ / a~ = PCA of text / audio fitted on the clips of the training rows (shared)
    tx = _PCA(PROD_K, random_state=0).fit(TXTn[tr].reshape(-1, TXTn.shape[-1]))
    am = AFDn[tr].reshape(-1) > 0
    ta = _PCA(PROD_K, random_state=0).fit(AUDn[tr].reshape(-1, AUDn.shape[-1])[am])

    def local(rows):
        out = np.zeros((len(rows), PROD_K * PROD_K))
        for k in range(3):
            xs, as_ = tx.transform(TXTn[rows, k]), ta.transform(AUDn[rows, k]) * AFDn[rows, k:k + 1]
            out += np.einsum('ni,nj->nij', xs, as_).reshape(len(rows), -1)
        return out
    D['local'] = (local(tr), local(te))
    # window: outer product of PCA-8 of the whole (standardised) X block and of the whole Z block
    W = []
    for blk in ('X', 'Z'):
        s = StandardScaler().fit(D[blk][0])
        p = _PCA(PROD_K, random_state=0).fit(s.transform(D[blk][0]))
        W.append((p.transform(s.transform(D[blk][0])), p.transform(s.transform(D[blk][1]))))
    D['window'] = tuple(np.einsum('ni,nj->nij', W[0][i], W[1][i]).reshape(len(W[0][i]), -1) for i in (0, 1))
    return D


LR_ARMS = {'Additive': ('X', 'Z'), 'Local': ('X', 'Z', 'local'), 'Window': ('X', 'Z', 'window')}


def lr_logits(Xtr, ytr, Xte, C):
    sc = StandardScaler().fit(Xtr)
    m = LogisticRegression(C=C, max_iter=5000).fit(sc.transform(Xtr), ytr)
    L = np.full((len(Xte), 7), np.nan)
    L[:, m.classes_] = m.decision_function(sc.transform(Xte))
    return np.where(np.isnan(L), np.nanmin(L, 1, keepdims=True) - 10, L)     # a class absent from training: very unlikely


LRP = {k: np.full((N, 7), np.nan) for k in LR_ARMS}
lr_log = []
for f in range(N_OUTER):
    tr, te = np.where(fold_of_row != f)[0], np.where(fold_of_row == f)[0]
    inner = {a: {C: np.zeros((len(tr), 7)) for C in LR_CS} for a in LR_ARMS}
    for ia, ib in GroupKFold(5).split(tr, groups=src[tr]):
        D = design(tr[ia], tr[ib])
        for a, blocks in LR_ARMS.items():
            Xa, Xb = (np.concatenate([D[b][i] for b in blocks], 1) for i in (0, 1))
            for C in LR_CS:
                inner[a][C][ib] = lr_logits(Xa, y_all[tr[ia]], Xb, C)
    D = design(tr, te)
    for a, blocks in LR_ARMS.items():
        scores = {C: nll(softmax_np(inner[a][C]), y_all[tr]) for C in LR_CS}
        C = min(scores, key=scores.get)
        t_ = fit_T(inner[a][C], y_all[tr])
        Xa, Xb = (np.concatenate([D[b][i] for b in blocks], 1) for i in (0, 1))
        LRP[a][te] = softmax_np(lr_logits(Xa, y_all[tr], Xb, C) / t_)
        lr_log.append({'fold': f, 'arm': a, 'C': C, 'T': t_, 'n_features': Xa.shape[1], 'inner_nll': scores[C]})
        print(f"fold {f} LR {a:<8}: C {C} | T {t_:.2f} | {Xa.shape[1]} features | NLL {nll(LRP[a][te], y_all[te]):.3f}",
              flush=True)
assert all(not np.isnan(v).any() for v in LRP.values())
pd.DataFrame(lr_log).to_csv(f"{OUT_DIR}/g20_lr_log.csv", index=False)
np.savez(f"{OUT_DIR}/g20_lr_oof_probs.npz", **LRP)

## Results and the fixed decision

In [ ]:
groups = [np.where(src == e)[0] for e in np.unique(src)]
rng = np.random.default_rng(0)
S_ = len(SEEDS)
DRAWS = [(np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))]), rng.integers(0, S_, S_))
         for _ in range(N_BOOT)]
ALL = np.arange(N)


def uar(P, idx):
    return war_uar(P[idx].argmax(1), y_all[idx], 7)[1]


def nn_stat(arm, idx, sd, fn):
    return fn(OOF[arm][sd].mean(0), idx) if fn is uar else nll(OOF[arm][sd].mean(0)[idx], y_all[idx])


def delta(fam, a, b, metric):
    # metric(a) - metric(b) on the pooled OOF predictions, with a paired bootstrap (two-level for the neural family)
    if fam == 'neural':
        f = lambda arm, idx, sd: nn_stat(arm, idx, sd, uar if metric == 'UAR' else None)
        pt = f(a, ALL, np.arange(S_)) - f(b, ALL, np.arange(S_))
        bs = [f(a, idx, sd) - f(b, idx, sd) for idx, sd in DRAWS]
    else:
        f = (lambda arm, idx: uar(LRP[arm], idx)) if metric == 'UAR' else (lambda arm, idx: nll(LRP[arm][idx], y_all[idx]))
        pt = f(a, ALL) - f(b, ALL)
        bs = [f(a, idx) - f(b, idx) for idx, _ in DRAWS]
    lo, hi = np.percentile(bs, [2.5, 97.5])
    return pt, lo, hi


print(f"== pooled out-of-fold scores ({N} MCIS, {len(EPS)} episodes; neural = mean of {S_} calibrated seeds) ==")
for k in NAMES:
    P = OOF[k].mean(0)
    print(f"  neural {k:<12} NLL {nll(P, y_all):.4f} | UAR {uar(P, ALL):5.2f}")
for k in LR_ARMS:
    print(f"  LR     {k:<12} NLL {nll(LRP[k], y_all):.4f} | UAR {uar(LRP[k], ALL):5.2f}")

ROWS = []
def row(test, fam, a, b, note):
    for metric in ('NLL', 'UAR'):
        pt, lo, hi = delta(fam, a, b, metric)
        if metric == 'UAR':                         # report UAR as b - a so that > 0 also means "b is better"
            pt, lo, hi = -pt, -hi, -lo
        ROWS.append({'test': test, 'family': fam, 'contrast': f"{a} vs {b}", 'metric': metric,
                     'delta': pt, 'lo': lo, 'hi': hi, 'note': note})
        print(f"  [{test}] {fam:<6} {metric}: Δ = {pt:+.4f} [{lo:+.4f}, {hi:+.4f}]  ({note})")


print("\n== main test: Δ_NLL = NLL(Additive) − NLL(Local); > 0 means the local interaction model is better ==")
for fam in ('neural', 'LR'):
    row('main', fam, 'Additive', 'Local', 'decision metric: NLL; UAR secondary')
print("\n== secondary: window variant ==")
for fam in ('neural', 'LR'):
    row('window', fam, 'Additive', 'Window', 'secondary')
print("\n== control: late fusion of separately trained single-source models (neural) ==")
row('control', 'neural', 'LateFusion', 'Additive', 'Δ > 0: joint additive training beats late fusion')
print("\n== T2: EMAP projection of full RoleNet; Δ = score(EMAP) − score(RoleNet) as NLL(EMAP) − NLL(RoleNet) ==")
row('EMAP', 'neural', 'Window-EMAP', 'Window', 'Δ > 0: the non-additive part of RoleNet adds forecast value')

R = pd.DataFrame(ROWS)
R.to_csv(f"{OUT_DIR}/g20_summary.csv", index=False)
main = R[(R.test == 'main') & (R.metric == 'NLL')].set_index('family')
passed = {fam: bool(main.loc[fam, 'lo'] > 0) for fam in ('neural', 'LR')}
n_pass = sum(passed.values())
dec = 'PILOT ALLOWED' if n_pass == 2 else 'STOP'
emap_row = R[(R.test == 'EMAP') & (R.metric == 'NLL')].iloc[0]
emap_drop = bool(emap_row.lo > 0)
print(f"\nmain test passed: neural {passed['neural']} | LR {passed['LR']}")
print(f"== G20 decision (fixed rule): {dec} ==")
if dec == 'STOP':
    print("   reading: no sufficiently strong evidence with these data and model classes (this does not show r = 0).")
if not emap_drop:
    print("   EMAP: no added forecast value of RoleNet's non-additive part has been seen"
          + (" -> gains of full RoleNet are not attributed to interaction." if dec != 'STOP' else "."))
json.dump({'decision': dec, 'main_pass': passed,
           'main': {fam: {k: float(main.loc[fam, k]) for k in ('delta', 'lo', 'hi')} for fam in ('neural', 'LR')},
           'emap_nll_delta': [float(emap_row.delta), float(emap_row.lo), float(emap_row.hi)], 'emap_drop': emap_drop},
          open(f"{OUT_DIR}/g20_decision.json", 'w'), indent=1)